In [1]:
import sys

sys.path.insert(
    0,
    "/Users/aditya/Documents/quantum-workspace/quantum-algorithms-toolkit"
)

from deutsch_jozsa import DeutschJoszaEngine
from bernstien_vazirani import BernsteinVazirani
from grover import Grover
from experiment import Experiment
from repeated_analysis import RepeatedAnalysis
from noise_model import HardwareNoiseModel
import random
from experiment_runner import ExperimentRunner
import pandas as pd
from qiskit_ibm_runtime.fake_provider import (
    FakeManilaV2,
    FakeLimaV2,
    FakeJakartaV2,
    FakeGuadalupeV2,
    FakeNairobiV2,
)

backends = [
    FakeManilaV2(),
    FakeLimaV2(),
    FakeJakartaV2(),
    FakeGuadalupeV2(),
    FakeNairobiV2(),
]

In [2]:
import pandas as pd
import random

results = []
algorithms = {
    "DJ": DeutschJoszaEngine,
    "BV": BernsteinVazirani,
    "Grover": Grover
}

for algorithm_name, algorithm_class in algorithms.items():
    for num_qubits in range(2, 5):
        if algorithm_name == "DJ":
            bitstring = None
            algorithm = algorithm_class(num_qubits)
        else:
            bitstring = ''.join(random.choice(['0', '1']) for _ in range(num_qubits))
            algorithm = algorithm_class(bitstring)

        qc = algorithm.Generate()
        noise = HardwareNoiseModel(backend=None)
        experiment = Experiment(
            name=f"{algorithm_name} - {num_qubits} Qubits",
            circuit=qc,
            noise_model=noise,
            shots=100
        )

        analysis = RepeatedAnalysis(experiment)
        stats = analysis.run(30)

        # separate run purely to capture one representative count distribution for display
        runner = ExperimentRunner()
        result = runner.run(experiment)

        results.append({
            "Algorithm": algorithm_name,
            "Qubits": num_qubits,
            "Bitstring": bitstring,
            "Mean TVD": stats["tvd_mean"],
            "TVD Std": stats["tvd_std"],
            "CoV": 100 * stats["tvd_std"]/stats["tvd_mean"],
            "N Runs": stats["runs"],
            "Ideal Counts": result.ideal_counts,
            "Noise Counts": result.noise_counts
        })

sweep_results = pd.DataFrame(results)
print(sweep_results)

  Algorithm  Qubits Bitstring  Mean TVD   TVD Std        CoV  N Runs  \
0        DJ       2      None  0.018667  0.012243  65.586189      30   
1        DJ       3      None  0.041667  0.020525  49.259342      30   
2        DJ       4      None  0.054000  0.022066  40.862468      30   
3        BV       2        10  0.045333  0.021453  47.322726      30   
4        BV       3       100  0.092333  0.023146  25.068100      30   
5        BV       4      0001  0.098000  0.033672  34.359025      30   
6    Grover       2        01  0.061667  0.023501  38.109794      30   
7    Grover       3       001  0.360333  0.050343  13.971093      30   
8    Grover       4      0111  0.825333  0.041996   5.088389      30   

                                        Ideal Counts  \
0                                         {'1': 100}   
1                                        {'00': 100}   
2                                       {'000': 100}   
3                                       {'001': 100}   

In [3]:
sweep_results.to_csv('generalsweep.csv')

In [4]:
import pandas as pd
import random

results = []
algorithms = {
    "DJ": DeutschJoszaEngine,
    "BV": BernsteinVazirani,
    "Grover": Grover
}

for backend in backends:
    for algorithm_name, algorithm_class in algorithms.items():
        for num_qubits in range(2, 5):
            if algorithm_name == "DJ":
                bitstring = None
                algorithm = algorithm_class(num_qubits)
            else:
                bitstring = ''.join(random.choice(['0', '1']) for _ in range(num_qubits))
                algorithm = algorithm_class(bitstring)

            qc = algorithm.Generate()
            noise = HardwareNoiseModel(backend)
            experiment = Experiment(
                name=f"{algorithm_name} - {num_qubits} Qubits - {backend.name}",
                circuit=qc,
                noise_model=noise,
                shots=100
            )

            analysis = RepeatedAnalysis(experiment)
            stats = analysis.run(30)

            runner = ExperimentRunner()
            result = runner.run(experiment)

            results.append({
                "Backend": backend.name,
                "Algorithm": algorithm_name,
                "Qubits": num_qubits,
                "Bitstring": bitstring,
                "Mean TVD": stats["tvd_mean"],
                "TVD Std": stats["tvd_std"],
                "CoV": 100 * stats["tvd_std"]/stats["tvd_mean"],
                "N Runs": stats["runs"],
                "Ideal Counts": result.ideal_counts,
                "Noise Counts": result.noise_counts
            })

sweep_results2 = pd.DataFrame(results)
print(sweep_results2)

           Backend Algorithm  Qubits Bitstring  Mean TVD   TVD Std  \
0      fake_manila        DJ       2      None  0.017333  0.013629   
1      fake_manila        DJ       3      None  0.064667  0.022087   
2      fake_manila        DJ       4      None  0.063000  0.021838   
3      fake_manila        BV       2        11  0.065000  0.023744   
4      fake_manila        BV       3       110  0.071333  0.024031   
5      fake_manila        BV       4      1001  0.114000  0.030917   
6      fake_manila    Grover       2        11  0.049000  0.022644   
7      fake_manila    Grover       3       101  0.454333  0.053799   
8      fake_manila    Grover       4      1101  0.884333  0.025554   
9        fake_lima        DJ       2      None  0.017667  0.011943   
10       fake_lima        DJ       3      None  0.056667  0.029981   
11       fake_lima        DJ       4      None  0.088667  0.025962   
12       fake_lima        BV       2        00  0.031333  0.015253   
13       fake_lima  

In [5]:
sweep_results2.to_csv('backend_sweep.csv')